In [1]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd
import numpy as np
from ipywidgets import interact
import ipywidgets as widgets
from sklearn.preprocessing import OneHotEncoder

In [4]:
hr = pd.read_csv("F:\MACHINE LEARNING\SOMETHING\Cases\human-resources-analytics\HR_comma_sep.csv")
# intervals=[(0.0,0.2),(50000,100000),(100000,150000),(150000,200000)]
hr.describe()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years
count,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000
mean,0.612890,0.716108,3.803001,201.050150,3.498166,0.144648,0.237879,0.021274
std,0.248615,0.171166,1.232266,49.943356,1.460262,0.351758,0.425799,0.144300
min,0.090000,0.360000,2.000000,96.000000,2.000000,0.000000,0.000000,0.000000
25%,0.440000,0.560000,3.000000,156.000000,3.000000,0.000000,0.000000,0.000000
50%,0.640000,0.720000,4.000000,200.000000,3.000000,0.000000,0.000000,0.000000
75%,0.820000,0.870000,5.000000,245.000000,4.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,7.000000,310.000000,10.000000,1.000000,1.000000,1.000000


In [5]:
def bin_col(input,b):
    binned = pd.cut(input, bins = b)
    return binned

In [6]:
hr['satisfaction'] = bin_col( hr['satisfaction_level'], 5)
hr['evaluation'] = bin_col( hr['last_evaluation'], 5)
hr['avg_hours'] = bin_col( hr['average_montly_hours'], 5)
hr['project'] = bin_col( hr['number_project'], 3)
hr['yrs_in_comp'] = bin_col( hr['time_spend_company'], 3)
hr.drop(['satisfaction_level','last_evaluation','average_montly_hours','number_project','time_spend_company'], axis = 1, inplace = True)

In [7]:
ohc = OneHotEncoder(sparse_output= False, dtype= np.bool_).set_output(transform = 'pandas')
fp_df = ohc.fit_transform(hr)

In [8]:
def gen_rules(min_support, min_confidence):
    itemsets = apriori(fp_df, min_support = min_support, use_colnames= True)
    rules = association_rules(itemsets, metric = 'confidence', min_threshold = min_confidence)
    rules = rules[['antecedents','consequents','support','confidence','lift']]
    relv_rules = rules[rules['lift'] > 1]
    return relv_rules.sort_values(by = ['confidence', 'lift', 'support'], ascending = False)

In [9]:
gen_rules(0.3,0.4)

,antecedents,consequents,support,confidence,lift
180,"(Work_accident_0, yrs_in_comp_(1.992, 4.667], ...",(promotion_last_5years_0),0.341581,0.993213,1.014802
62,"(Work_accident_0, salary_low)",(promotion_last_5years_0),0.415138,0.992348,1.013918
134,"(yrs_in_comp_(1.992, 4.667], salary_low)",(promotion_last_5years_0),0.398866,0.991874,1.013434
27,(salary_low),(promotion_last_5years_0),0.483294,0.990975,1.012515
102,"(left_0, salary_low)",(promotion_last_5years_0),0.339580,0.989891,1.011408
...,...,...,...,...,...
198,"(promotion_last_5years_0, yrs_in_comp_(1.992, ...","(Work_accident_0, project_(1.995, 3.667])",0.327776,0.408970,1.113179
137,(promotion_last_5years_0),"(yrs_in_comp_(1.992, 4.667], salary_low)",0.398866,0.407536,1.013434
186,"(Work_accident_0, promotion_last_5years_0)","(yrs_in_comp_(1.992, 4.667], salary_low)",0.341581,0.407057,1.012242
99,"(yrs_in_comp_(1.992, 4.667])","(Work_accident_0, project_(1.995, 3.667])",0.331911,0.406584,1.106686
